# Assignment 2

**Credits**: Federico Ruggeri, Giulia Grundler, Paolo Torroni

**Keywords**: Fallacy Detection, Multi-label Classification, LLMs, Prompting

<style>
/* Long code lines wrap instead of running off the right edge of the slide,
   and a slide taller than the 700px deck scrolls instead of being cut. */
.reveal .jp-RenderedHTMLCommon pre,
.reveal .highlight pre,
.reveal pre, .reveal pre code {
  white-space: pre-wrap;
  word-break: break-word;
  box-sizing: border-box;
  max-width: 100%;
}
.reveal .slides section:not(.stack) {
  max-height: 700px;
  overflow-y: auto;
  overflow-x: hidden;
}
</style>

# Contact

For any doubt, question, issue or help, you can always contact us at the following email addresses:

Teaching Assistants:

* Federico Ruggeri -> federico.ruggeri6@unibo.it
* Giulia Grundler -> giulia.grundler2@unibo.it

# Relevant Material

- Huggingface documentation
- Huggingface hub

# Introduction

You are tasked to address **fallacy detection** on
[MAFALDA](https://github.com/ChadiHelwe/MAFALDA), a benchmark that unifies several older fallacy corpora under a
single taxonomy.

A **fallacy** is an argument whose premises do not entail its conclusion. Fallacies are everywhere in online
discussion, and spotting them is a task both humans and models find hard.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


## Preliminaries

### Huggingface Login

Once we have created an account and an access token, we need to login to Huggingface via code.

- Paste your token in the login box and press Enter

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

After login, you can download all models associated with your access token in addition to those that are not
protected by an access token.

### Data Loading

Since we are only interested in prompting, we do not require a train dataset.

We have prepared a version of MAFALDA in our dedicated
[Github repository](https://github.com/lt-nlp-lab-unibo/nlp-course-material).

Check the ``Assignment 2/data`` folder.
It contains:

- ``a2_test.csv`` -> a test set of 156 texts.
- ``demonstrations.csv`` -> a pool of 44 texts for few-shot prompting.

### Columns

Both files have the same four columns:

- ``mafalda_id`` -> the line of the text in the original MAFALDA release.
- ``text`` -> the text to classify.
- ``labels_level1`` -> the categories of the text, semicolon-separated, **empty** when the text has no fallacy.
- ``labels_level2`` -> the fine-grained fallacy types, semicolon-separated. Only needed for the bonus.

### Label distribution

| file | texts | no fallacy | credibility | emotion | logic |
|---|---|---|---|---|---|
| `a2_test.csv` | 156 | 50 | 42 | 28 | 76 |
| `demonstrations.csv` | 44 | 13 | 13 | 10 | 22 |

Columns do not sum to the number of texts: a text can carry several categories.

The corpus is small and `emotion` is the rarest category, so expect noisy numbers. Say so in your report rather than
reading too much into a two-point gap.

### Instructions

We require you to:

* **Download** the ``Assignment 2/data`` folder.
* **Encode** ``a2_test.csv`` into a ``pandas.DataFrame`` object.
* **Convert** ``labels_level1`` into a 3-dimensional binary vector, in the order
  ``[credibility, emotion, logic]``.

In [ ]:
CLASSES = ["credibility", "emotion", "logic"]

df = pd.read_csv("data/a2_test.csv")
df["labels_level1"] = df["labels_level1"].fillna("").apply(
    lambda s: [int(c in s.split(";")) for c in CLASSES]
)

print(df.head())

   mafalda_id                                               text  \
0           1  Two of my best friends are really introverted,...   
1           2  TITLE: There is a difference between a'smurf' ...   
2           4  America is the best place to live, because it'...   
3           7  "Four out of five dentists recommend Happy Glo...   
4           8  TITLE: Gatineau police officer tests positive,...   

  labels_level1                labels_level2  
0     [0, 0, 1]         hasty generalization  
1     [0, 0, 1]                false analogy  
2     [0, 0, 1]           circular reasoning  
3     [1, 0, 0]  appeal to (false) authority  
4     [0, 0, 1]               slippery slope  


# [Task 1 - 0.25 points] Model setup

Once the test data has been loaded, we have to setup the model pipeline for inference.

In particular, we have to:
- Load the model weights from Huggingface
- Quantize the model to fit into a single-GPU limited hardware

In [ ]:

model_cards = [
    "Qwen/Qwen3-4B-Instruct-2507",
    "microsoft/Phi-4-mini-instruct"
]
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 has no bf16
)

models = {}
for card in model_cards:
    tokenizer = AutoTokenizer.from_pretrained(card)
    model = AutoModelForCausalLM.from_pretrained(
        card, quantization_config=quant_config, device_map="auto"
    )
    models[card] = (model, tokenizer)
    print(card, f"{model.get_memory_footprint() / 1e9:.2f} GB")


# [Task 2 - 0.5 points] Prompt setup

Prompting requires an input pre-processing phase where we convert each input example into a specific instruction
prompt.

## Prompt Template

Use the following prompt template to process input texts.

In [ ]:
DEFINITIONS = '''Below you find the category definitions:
Credibility: the argument relies on who is speaking, or on who else agrees, instead of on evidence.
Logic: the reasoning is broken, the conclusion does not follow from the premises.
Emotion: the argument works on the feelings of the reader instead of on the merits of the claim.'''

The definitions above go inside the instructions, which are the same in Task 2 and Task 5.

In [ ]:
INSTRUCTIONS = '''Your task is to detect fallacies in the input text.
A fallacy is an argument whose premises do not entail its conclusion.
Classify the input text according to which of the following three
categories of fallacy it contains.

{definitions}

A text may contain more than one category, or none at all.

Respond only by writing the categories that apply, separated by a comma,
among the following: credibility, logic, emotion.
Respond only by writing none if the text contains no fallacy.'''

The prompt itself is the instructions, plus the text to classify.

In [ ]:
prompt = [
    {
        'role': 'system',
        'content': 'You are an annotator for fallacy detection.'
    },
    {
        'role': 'user',
        'content': INSTRUCTIONS.format(definitions=DEFINITIONS) + '''

TEXT: {text}

ANSWER:
'''
    }
]

### Instructions

In order to get Task 2 points, we require you to:

* Write a ``prepare_prompts`` function as the one reported below.

In [ ]:
def prepare_prompts(texts, prompt_template, tokenizer):
    """This function format input text samples into instructions prompts.

    Inputs:
      texts: input texts to classify via prompting
      prompt_template: the prompt template provided in this assignment
      tokenizer: the transformers Tokenizer object instance associated
      with the chosen model card

    Outputs:
      input texts to classify in the form of instruction prompts
    """

    prompts = []
    for text in texts:
        messages = [
            {
                "role": m["role"],
                "content": m["content"].replace("{text}", text)
            }
            for m in prompt_template
        ]
        prompts.append(tokenizer.apply_chat_template(
            messages,
            # dont tokenize text yet
            tokenize=False,
            # adds assistant marker to trigger a response instead of continuation
            add_generation_prompt=True 
        ))

    return prompts

# [Task 3 - 0.5 points] Inference

We are now ready to define the inference loop where we prompt the model with each pre-processed sample.

### Instructions

In order to get Task 3 points, we require you to:

* Write a ``generate_responses`` function as the one reported below.
* Write a ``process_response`` function as the one reported below.

In [ ]:
def generate_responses(model, tokenizer, prompt_examples):
    """This function implements the inference loop for a LLM model.
    Given a set of examples, the model is tasked to generate
    a response.

    Inputs:
      model: LLM model instance for prompting
      tokenizer: the transformers Tokenizer object instance associated
      with the chosen model card
      prompt_examples: pre-processed text samples

    Outputs:
      the generated responses, as strings, one per input example
    """
    
    responses = []
    for e in prompt_examples:
        inputs = tokenizer(
            e, return_tensors="pt",
            # special tokens already added during preprocessing
            add_special_tokens=False
        ).to(model.device)
        output = model.generate(
            **inputs,
            max_new_tokens=32,
            # greedy decoding for reproducibility, worth to reevaluate
            do_sample=False
        )
        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        responses.append(tokenizer.decode(
            new_tokens,
            # avoid EOS/EOT. theyre not consistent and i dont want to parse them away myself
            skip_special_tokens=True
        ))
    
    return responses

In [ ]:
def process_response(response):
    """This function takes a textual response generated by the LLM
    and processes it to map the response to a multi-label vector.

    Inputs:
      response: generated response from LLM

    Outputs:
      a 3-dimensional binary vector, in the order
      [credibility, emotion, logic], or None when the response
      mentions no category and does not say none.
    """
    response = response.lower()
    vector = [int(c in response) for c in CLASSES]

    if any(vector):
        return vector
    if "none" in response:
        return [0, 0, 0]
    return None

# [Task 4 - 0.25 points] Metrics

In order to evaluate selected LLMs, we need to compute performance metrics.

We compute **macro F1-score** and the ratio of failed responses generated by models (**fail-ratio**).

That is, how frequent the LLM fails to follow instructions and provides incorrect responses that do not address the
classification task.

Since the task is multi-label, macro F1 is the average of the three **binary** F1-scores, one per category, each
computed on the positive class.

Failed responses are counted in the fail-ratio and treated as ``[0, 0, 0]`` when computing F1.

### Instructions

In order to get Task 4 points, we require you to:

* Write a ``compute_metrics`` function as the one reported below.
* Compute metrics for the two selected LLMs.
* Report the **per-category** F1-score alongside the macro average.

In [ ]:
def compute_metrics(responses, y_true):
    """This function takes the generated responses and the ground-truth
    labels and computes metrics. In particular, this function computes
    macro f1-score and fail-ratio metrics. It internally invokes
    `process_response`, since parsing is part of what we measure.

    Inputs:
      responses: the raw responses generated by the LLM, as strings
      y_true: ground-truth multi-label vectors

    Outputs:
      dictionary containing desired metrics
    """

    y_pred = [process_response(r) for r in responses]
    fail_ratio = sum(p is None for p in y_pred) / len(y_pred)
    y_pred = [p if p is not None else [0, 0, 0] for p in y_pred]
    f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

    return {
        "f1_credibility": f1[0],
        "f1_emotion": f1[1],
        "f1_logic": f1[2],
        "macro_f1": f1.mean(),
        "fail_ratio": fail_ratio,
        # "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        # "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

In [ ]:
texts = df["text"].tolist()
y_true = df["labels_level1"].tolist()

results = {}
for card, (model, tokenizer) in models.items():
    prompts = prepare_prompts(texts, prompt, tokenizer)
    responses = generate_responses(model, tokenizer, prompts)
    results[card] = compute_metrics(responses, y_true)

In [ ]:

res = pd.DataFrame(results).T
res.index = [card.split("/")[-1] for card in res.index]
colors = ["#2a78d6", "#eb6834"]

def bar_chart(ax, data, title):
    """Grouped bars: one group per metric, one bar per model."""
    x = np.arange(len(data.columns))
    width = 0.8 / len(data)
    for i, (name, row) in enumerate(data.iterrows()):
        offset = (i - (len(data) - 1) / 2) * width
        bars = ax.bar(x + offset, row, width, label=name, color=colors[i], edgecolor="white", linewidth=2)
        ax.bar_label(bars, fmt="%.2f", fontsize=8)
    if len(data.columns) > 1:
        ax.set_xticks(x, data.columns)
    else:
        ax.set_xticks([])
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
bar_chart(axes[0], res[["macro_f1"]], "Macro F1")
bar_chart(axes[1], res[["fail_ratio"]], "Fail ratio")
axes[0].legend()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
bar_chart(ax, res.drop(columns=["macro_f1", "fail_ratio"]), "Other metrics")
ax.legend()
plt.show()

# [Task 5 - 0.5 points] Few-shot Inference

So far, we have tested models in a zero-shot fashion: we provide the input text to classify and instruct the model to
generate a response.

We are now interested in performing few-shot prompting to see the impact of providing demonstration examples.

To do so, we slightly change the prompt template as follows.

In [ ]:
few_shot_prompt = [
    {
        'role': 'system',
        'content': 'You are an annotator for fallacy detection.'
    },
    {
        'role': 'user',
        'content': INSTRUCTIONS.format(definitions=DEFINITIONS) + '''

EXAMPLES: {examples}

TEXT: {text}

ANSWER:
'''
    }
]

The new prompt template reports some demonstration examples to instruct the model.

Generally, we provide an equal number of demonstrations per class as shown in the example below.

In [ ]:
EXAMPLES:
TEXT: **example 1**
ANSWER: logic
TEXT: **example 2**
ANSWER: credibility, emotion
TEXT: **example 3**
ANSWER: none

## Instructions

In order to get Task 5 points, we require you to:

- Load ``demonstrations.csv`` and encode it into a ``pandas.DataFrame`` object.
- Define a ``build_few_shot_demonstrations`` function as the one reported below.
- Modify ``prepare_prompts`` to support demonstrations.
- Perform few-shot inference as in Task 3.
- Compute metrics as in Task 4.

In [ ]:
def build_few_shot_demonstrations(demonstrations, num_per_class=2):
  """
    Inputs:
      demonstrations: DataFrame wrapping demonstrations.csv
      num_per_class: number of demonstrations per class

    Outputs:
      list of demonstrations to inject into the prompt template.
  """
  pass

## Notes

1. You are free to pick any value for ``num_per_class``.

2. Demonstrations are multi-label too. Decide whether "one demonstration per class" means one text **containing**
   that category, or one text carrying **only** that category, and say which you chose. Do not forget texts with no
   fallacy: a model that never sees ``none`` will rarely answer it.

3. According to our tests, few-shot prompting increases inference time by some minutes (we experimented with
   ``num_per_class`` $\in [2, 4]$). MAFALDA texts are long, so keep an eye on the context window.

# [Task 6 - 1.0 points] Error Analysis

**For this task, the use of AI is prohibited. Any use constitutes a violation of academic integrity.**

We are now interested in evaluating model responses and comparing their performance.

This analysis helps us in understanding

- Classification task performance gap: are the models good at this task?
- Generation quality: which kind of responses do models generate?
- Errors: which kind of mistakes do models do?

### Instructions

In order to get Task 6 points, we require you to:

* Compare classification performance of selected LLMs in a Table.
* Compute a confusion matrix **per category** for selected LLMs.
* Briefly summarize your observations on generated responses.

### Suggestions

Some questions worth answering:

* Do the models over-predict? Compare how many categories they assign per text against the 0.94 of the gold standard.
* Is `emotion` harder than `logic`, and is that the task or the 28 test texts that carry it?
* Do models catch the fallacy-free texts, or do they find a fallacy in everything?
* Do few-shot demonstrations fix the failures, or only the formatting of the answers?

# [Task 7 - 1.0 points] Report

**For this task, the use of AI is prohibited. Any use constitutes a violation of academic integrity.**

Wrap up your experiment in a short report (up to 2 pages).

### Instructions

* Use the NLP course report template.
* Summarize each task in the report following the provided template.

### Recommendations

The report is not a copy-paste of graphs, tables, and command outputs.

* Summarize classification performance in Table format.
* **Do not** report command outputs or screenshots.
* The error analysis section should summarize your findings.

# Submission

* **Submit** your report in PDF format.
* **Submit** your python notebook.
* Make sure your notebook is **well organized**, with no temporary code, commented sections, tests, etc...

# FAQ

Please check this frequently asked questions before contacting us.

### Use of AI

You are free to use AI as a support in your solution of the assignment, except for **Error Analysis** and
**Report**.

### Group work

Does working alone or in group affect assignment scoring?

No, it is not part of the evaluation.

### Model cards

You can pick any open-source model card you like.

We recommend starting from those reported in this assignment.

### Implementation

Everything can be done via ``transformers`` APIs.

However, you are free to test frameworks, such as [LangChain](https://www.langchain.com/),
[LlamaIndex](https://www.llamaindex.ai/), provided that you correctly address task instructions.

### Task Performance

The task is challenging and zero-shot prompting may show relatively low performance depending on the chosen model.

The MAFALDA authors report that LLMs are still far from human performance on this benchmark, so low scores are an
expected result to be analysed, not a bug to be hidden.

### Prompt Template

Do not change the provided prompt template.

You are only allowed to change it in case of a possible extension.

### Optimizations

Any kind of code optimization (e.g., speedup model inference or reduce computational cost) is more than welcome!

### Bonus Points

0.5 bonus points are arbitrarily assigned based on significant contributions such as:

- Outstanding error analysis
- Masterclass code organization
- **Level 2 fallacy detection**: repeat the experiment on the ``labels_level2`` column, the 23 fine-grained fallacy
  types of MAFALDA. Beware that some of them appear in a handful of texts.
- Perform prompt tuning
- **Study open-source LLMs served through public APIs**, such as
  [Gemma](https://ai.google.dev/gemma), instead of running the weights locally.

Note that bonus points are only assigned if all task points are attributed (i.e., 4/4).

### Dataset Reference

Chadi Helwe, Tom Calamai, Pierre-Henri Paris, Chloé Clavel, Fabian Suchanek. 2024.
*MAFALDA: A Benchmark and Comprehensive Study of Fallacy Detection and Classification*. NAACL 2024.

The corpus is released under CC BY-SA 4.0.

# The End

Feel free to reach out for questions/doubts!